In [2]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import optuna
import random
import mlflow

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate, cross_val_predict
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report, accuracy_score, f1_score, brier_score_loss, auc
from sklearn.preprocessing import StandardScaler, label_binarize, LabelEncoder
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.calibration import calibration_curve
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline


import xgboost as xgb

from itertools import product

pd.set_option('display.max_rows', 1000)

In [3]:
mlflow.set_tracking_uri("http://127.0.0.1:5000")
mlflow.set_experiment("btc_timezone_analysis_model_results")

<Experiment: artifact_location='/home/matej/btc_timezone_analysis/model_training/mlruns_artifacts/2', creation_time=1788765291756, effective_trace_archival_retention=None, experiment_id='2', last_update_time=1788765291756, lifecycle_stage='active', name='btc_timezone_analysis_model_results', tags={}, trace_location=None, workspace='default'>

# Functions

In [4]:
def class_report(y_train, y_train_pred, y_test, y_test_pred, target_name):
    # Definition of function which produce classification report and confusion matrix for train and also test set 
    fig, axes = plt.subplots(nrows=2, ncols=2, figsize=(15, 12), 
                             gridspec_kw={'height_ratios': [0.5, 1.5]})
    
    # Train classification report
    axes[0, 0].text(-0.2, 0.4, classification_report(y_train, y_train_pred, target_names=["America", "Central_Asia", "East_Asia_Pac", "Euro_Africa"]), 
                    fontsize=13, family='monospace', va='center')
    axes[0, 0].set_title("Classification Report (Train)", fontsize=14, fontweight='bold')
    axes[0, 0].axis('off') 
    
    # Test classification report
    axes[0, 1].text(-0.2, 0.4, classification_report(y_test, y_test_pred, target_names=["America", "Central_Asia", "East_Asia_Pac", "Euro_Africa"]), 
                    fontsize=13, family='monospace', va='center')
    axes[0, 1].set_title("Classification Report (Test)", fontsize=14, fontweight='bold')
    axes[0, 1].axis('off') 
    
    # Train Confusion Matrix
    ConfusionMatrixDisplay.from_predictions(
        y_train, y_train_pred, 
        ax=axes[1, 0], cmap='Blues', normalize='true',
        labels=target_name
    )
    axes[1, 0].set_title("Training Confusion Matrix (Normalized)")
    axes[1, 0].tick_params(axis='x', rotation=90)
    
    # Test Confusion Matrix
    ConfusionMatrixDisplay.from_predictions(
        y_test, y_test_pred, 
        ax=axes[1, 1], cmap='Greens', normalize='true',
        labels=target_name
    )
    axes[1, 1].set_title("Testing Confusion Matrix (Normalized)")
    axes[1, 1].tick_params(axis='x', rotation=90)
    
    plt.savefig("./plots/class_report", dpi=300, bbox_inches='tight')
    plt.close()

In [5]:
def calibration_curve_classes(y_train, y_train_proba, y_train_pred, y_test, y_test_proba, y_test_pred, colors, labels):
    # Definition of function which produce calibration curve per each region for train and also test set 
    fig, ax = plt.subplots(1, 2, figsize=(15, 6))
    
    for j, a in zip(range(2), ["train", "test"]):
        ax[j].plot([0, 1], [0, 1], "k--", label="Perfectly Calibrated")
        ax[j].plot([0.75, 0.75], [-0.05, 1.05], "--", color="#C0C0C0")
        for l, c in zip(labels, colors):
            proba = locals()[f"y_{a}_proba"]
            pred  = locals()[f"y_{a}_pred"]
            true  = locals()[f"y_{a}"]
        
            subset = (true == l)
            
            confidences = np.max(proba[subset], axis=1)
            accuracies = (pred[subset] == true[subset]).astype(int)
            n_bins = 5
            percentiles = np.linspace(0.0, 1.0, n_bins + 1)
            bins = np.quantile(confidences, percentiles)
            bin_indices = np.digitize(confidences, bins) - 1
            bin_indices[bin_indices == n_bins] = n_bins - 1
            
            bin_accuracies = []
            bin_confidences = []
            bin_percentages = []
            
            for i in range(n_bins):
                mask = (bin_indices == i)
                if np.sum(mask) > 0: # Ak sú v bine nejaké dáta
                    bin_accuracies.append(np.mean(accuracies[mask]))
                    bin_confidences.append(np.mean(confidences[mask]))
                    bin_percentages.append((np.sum(mask) / len(confidences)) * 100)
        
            ax[j].plot(bin_confidences, bin_accuracies, marker='o', linestyle='-', color=c, label=l)
        
        ax[j].set_xlabel("Confidence")
        ax[j].set_ylabel("True Accuracy")
        ax[j].set_title(f"Calibration Graph per each class on {a} set")
        ax[j].legend(loc="upper left")
        ax[j].grid(True, alpha=0.5)
        ax[j].set_xlim([-0.05, 1.05])
        ax[j].set_ylim([-0.05, 1.05])
        
    plt.savefig("./plots/calibration_graph.png", dpi=300, bbox_inches='tight')
    plt.close()

In [6]:
def get_mlflow_confidence_metrics(
    y_test_true, y_test_pred, y_test_prob, 
    y_train_true, y_train_pred, y_train_prob, 
    target_names
):
    metrics = {}

    metrics['train_brier_score'] = float(brier_score_loss(label_binarize(y_train_true, classes=target_names), y_train_prob))
    metrics['test_brier_score'] = float(brier_score_loss(label_binarize(y_test_true, classes=target_names), y_test_prob))
    metrics['train_accuracy'] = float(accuracy_score(y_train_true, y_train_pred))
    metrics['test_accuracy'] = float(accuracy_score(y_test_true, y_test_pred))
    metrics['samples'] = len(y_test_true)

    
    # 3. Confidence Thresholds (Calculated on the Test set)
    y_test_true = np.array(y_test_true)
    y_test_pred = np.array(y_test_pred)
    confidences = np.max(np.array(y_test_prob), axis=1)
    
    thresholds = [0.50, 0.60, 0.70, 0.80, 0.90]
    
    for t in thresholds:
        t_str = f"test_conf_{int(t * 100)}"
        mask = (confidences >= t)
        n_samples = int(np.sum(mask))
        
        metrics[f"{t_str}_samples"] = n_samples
        if n_samples > 0:
            metrics[f"{t_str}_accuracy"] = float(accuracy_score(y_test_true[mask], y_test_pred[mask]))
        else:
            metrics[f"{t_str}_accuracy"] = 0.0
            
    return metrics

In [7]:
def calculate_aurc(y_true, y_prob, y_pred):
    """
    Calculates the Area Under the Risk-Coverage Curve (AURC).
    """
    # 1. Convert to pure NumPy arrays to avoid Pandas KeyErrors
    y_true_np = np.array(y_true)
    y_pred_np = np.array(y_pred)
    
    # 2. Extract the model's confidence
    confidences = np.max(y_prob, axis=1)
    
    # 3. Determine which predictions are incorrect (Error = 1, Correct = 0)
    # By using y_pred directly, this safely compares strings to strings or ints to ints!
    errors = (y_true_np != y_pred_np).astype(int)
    
    # 4. Sort the samples by confidence in descending order
    sorted_indices = np.argsort(-confidences)
    sorted_errors = errors[sorted_indices]
    
    # 5. Calculate Coverage (x-axis)
    n_samples = len(y_true_np)
    coverages = np.arange(1, n_samples + 1) / n_samples
    
    # 6. Calculate Risk (y-axis): The cumulative mean of errors
    cumulative_errors = np.cumsum(sorted_errors)
    risks = cumulative_errors / np.arange(1, n_samples + 1)
    
    # 7. Add the starting point (0 coverage) to anchor the graph
    coverages = np.insert(coverages, 0, 0.0)
    risks = np.insert(risks, 0, risks[0]) 
    
    # 8. Calculate the area under the curve using the trapezoidal rule
    aurc_score = auc(coverages, risks)
    
    return coverages, risks, aurc_score

In [8]:
def plot_risk_coverage_curve(y_train, y_train_proba, y_train_pred, y_test, y_test_proba, y_test_pred):
    """Plots the Risk-Coverage curves for Train and Test sets side-by-side."""
    fig, axes = plt.subplots(1, 2, figsize=(15, 6))
    
    # Bundle the datasets to iterate through them easily
    datasets = [
        ("Train", y_train, y_train_proba, y_train_pred, axes[0]),
        ("Test", y_test, y_test_proba, y_test_pred, axes[1])
    ]
    
    for name, true, proba, pred, ax in datasets:
        # Get curve coordinates and score (Passing 'pred' here to fix the string mismatch)
        coverages, risks, aurc_score = calculate_aurc(true, proba, pred)
        
        # Plot the actual curve
        ax.plot(coverages, risks, color='blue', linewidth=2, label=f'Model AURC = {aurc_score:.4f}')
        
        # Plot the baseline (Full dataset error)
        # Converted to np.array to prevent Pandas index mismatches here as well
        baseline_error = np.mean(np.array(true) != np.array(pred))
        ax.axhline(y=baseline_error, color='red', linestyle='--', label=f'Full Dataset Error ({baseline_error:.4f})')
        
        # Formatting
        ax.set_title(f"Risk-Coverage Curve ({name} Set)", fontsize=14, fontweight='bold')
        ax.set_xlabel("Coverage (Proportion of dataset evaluated)")
        ax.set_ylabel("Risk (Error rate on covered samples)")
        ax.legend(loc="upper left")
        ax.grid(True, alpha=0.5)
        ax.set_xlim([-0.05, 1.05])
        
    plt.savefig("./plots/risk_coverage.png", dpi=300, bbox_inches="tight")
    plt.close()

# Data

In [9]:
#Importing data from csv, the path could be different
data = pd.read_csv("/home/matej/btc_timezone_analysis/data_preprocessing/database_from_bitcointalk.csv")
len(data)

1941

In [10]:
data.columns = data.columns.astype(str)
data.head()

,Unnamed: 0,entity,offset,0,1,2,3,4,5,6,...,17_weekend,18_weekend,19_weekend,20_weekend,21_weekend,22_weekend,23_weekend,total_weekday,total_weekend,region
0,0,Entity 100,7.0,0.020921,0.037657,0.025105,0.050209,0.050209,0.058577,0.033473,...,0.197674,0.046512,0.011628,0.023256,0.023256,0.011628,0.023256,153.0,62.0,East_Asia_Pac
1,1,Entity 1000,2.0,0.107143,0.107143,0.071429,0.035714,0.017857,0.035714,0.053571,...,0.033333,0.033333,0.033333,0.033333,0.033333,0.033333,0.033333,26.0,6.0,Euro_Africa
2,2,Entity 1001,2.0,0.034483,0.034483,0.051724,0.051724,0.068966,0.068966,0.103448,...,0.076923,0.038462,0.038462,0.038462,0.038462,0.038462,0.038462,32.0,2.0,Euro_Africa
3,3,Entity 1002,-5.0,0.053571,0.071429,0.017857,0.017857,0.017857,0.017857,0.017857,...,0.028571,0.028571,0.028571,0.057143,0.028571,0.057143,0.142857,21.0,11.0,America
4,5,Entity 1004,7.0,0.017241,0.017241,0.017241,0.051724,0.086207,0.068966,0.086207,...,0.034483,0.034483,0.034483,0.068966,0.034483,0.034483,0.034483,29.0,5.0,East_Asia_Pac


In [11]:
X = data
y = data["region"]
col_names = ["0", "1", "2", "3", "4", "5", "6", "7", "8", "9", "10", "11", "12", "13", "14", "15", "16", "17", "18", "19", "20", "21", "22", "23"]

# Logistic Regression one model

In [ ]:
test_size = 0.25
target_names = ["America", "Euro_Africa", "Central_Asia", "East_Asia_Pac"]
colors = ['red', 'green', 'blue', 'black']
num = 1019

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, stratify = y, random_state = num)
X_train = X_train[col_names]
X_test = X_test[col_names]
sample_weights = compute_sample_weight(class_weight='balanced', y=y_train)


with mlflow.start_run(run_name = "logistic_regression_model"):
    mlflow.set_tag("model_type", "logistic regression test")
    mlflow.set_tag("data type", "24 hour transaction distribution")
    pipeline = Pipeline([
        ('scaler', StandardScaler()),
        ('log_reg', LogisticRegression(max_iter=1000, class_weight='balanced'))
    ])
    pipeline.fit(X_train, y_train)

    y_train_pred = np.array(pipeline.predict(X_train))
    y_train_proba = np.array(pipeline.predict_proba(X_train))

    y_test_pred = np.array(pipeline.predict(X_test))
    y_test_proba = np.array(pipeline.predict_proba(X_test))

    metrics = get_mlflow_confidence_metrics(
        y_test, y_test_pred, y_test_proba, 
        y_train, y_train_pred, y_train_proba, 
        pipeline.classes_
    )
    mlflow.log_metrics(metrics)
    class_report(y_train, y_train_pred, y_test, y_test_pred, target_names)
    calibration_curve_classes(y_train, y_train_proba, y_train_pred, y_test, y_test_proba, y_test_pred, colors, target_names)
    plot_risk_coverage_curve(y_train, y_train_proba, y_train_pred, y_test, y_test_proba, y_test_pred)
    mlflow.log_artifact("./plots/class_report.png", artifact_path = "plots")
    mlflow.log_artifact("./plots/calibration_graph.png", artifact_path = "plots")
    mlflow.log_artifact("./plots/risk_coverage.png", artifact_path = "plots")

🏃 View run logistic_regression_model at: http://127.0.0.1:5000/#/experiments/2/runs/3de00664b12441c689bea9b7c9d76767
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2


# Logistic regression bootsraping

In [12]:

n_iterations = 1000
test_size = 0.25
target_names = ["America", "Euro_Africa", "Central_Asia", "East_Asia_Pac"]
colors = ['red', 'green', 'blue', 'black']



with mlflow.start_run(run_name = "logistic_regression_model"):
    mlflow.set_tag("model_type", "logistic regression test bootstrapped")
    mlflow.set_tag("data type", "24 hour transaction distribution")
    pipeline = Pipeline([
        ('scaler', StandardScaler()),
        ('log_reg', LogisticRegression(max_iter=1000, class_weight='balanced'))
    ])

    y_test_boot = []
    y_test_boot_pred = []
    y_test_boot_proba = []

    y_train_boot = []
    y_train_boot_pred = []
    y_train_boot_proba = []

    all_metrics = []
    all_class_reports = []

    for i in range(n_iterations):
        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, stratify = y)
        X_train = X_train[col_names]
        X_test = X_test[col_names]
        sample_weights = compute_sample_weight(class_weight='balanced', y=y_train)
        pipeline.fit(X_train, y_train)

        y_train_pred = np.array(pipeline.predict(X_train))
        y_train_proba = np.array(pipeline.predict_proba(X_train))

        y_test_pred = np.array(pipeline.predict(X_test))
        y_test_proba = np.array(pipeline.predict_proba(X_test))

        y_train_boot.extend(y_train)
        y_train_boot_pred.extend(y_train_pred)
        y_train_boot_proba.extend(y_train_proba)

        y_test_boot.extend(y_test)
        y_test_boot_pred.extend(y_test_pred)
        y_test_boot_proba.extend(y_test_proba)

        metrics = get_mlflow_confidence_metrics(
            y_test, y_test_pred, y_test_proba, 
            y_train, y_train_pred, y_train_proba, 
            pipeline.classes_
        )
        all_metrics.append(metrics)

        report_dict = classification_report(y_test, y_test_pred, target_names=target_names, output_dict=True, zero_division=0)
        
        # Flatten the dictionary so it can be easily converted to a DataFrame later
        flat_report = {}
        for label, class_metrics in report_dict.items():
            if isinstance(class_metrics, dict):
                for metric_name, val in class_metrics.items():
                    flat_report[f"{label}_{metric_name}"] = val
            else:
                flat_report[f"{label}"] = class_metrics
                
        all_class_reports.append(flat_report)
    
    y_train_boot = np.array(y_train_boot)
    y_train_boot_pred = np.array(y_train_boot_pred)
    y_train_boot_proba = np.array(y_train_boot_proba)   
    y_test_boot = np.array(y_test_boot) 
    y_test_boot_pred = np.array(y_test_boot_pred)
    y_test_boot_proba = np.array(y_test_boot_proba)

    # Convert the list of dictionaries into a Pandas DataFrame
    df_reports = pd.DataFrame(all_class_reports)
    
    # Calculate Mean, 2.5%, and 97.5% quantiles across the 1000 runs
    report_summary = pd.DataFrame({
        'mean': df_reports.mean(),
        'quantile_2.5%': df_reports.quantile(0.025),
        'quantile_97.5%': df_reports.quantile(0.975)
    })
    
    # Save the quantiles to a CSV and log to MLflow
    csv_path = "./plots/classification_report_quantiles.csv"
    report_summary.to_csv(csv_path)
    mlflow.log_artifact(csv_path, artifact_path="metrics")


    metrics = get_mlflow_confidence_metrics(
        y_test_boot, y_test_boot_pred, y_test_boot_proba, 
        y_train_boot, y_train_boot_pred, y_train_boot_proba, 
        pipeline.classes_
    )
    mlflow.log_metrics(metrics)
    class_report(y_train_boot, y_train_boot_pred, y_test_boot, y_test_boot_pred, target_names)
    calibration_curve_classes(y_train_boot, y_train_boot_proba, y_train_boot_pred, y_test_boot, y_test_boot_proba, y_test_boot_pred, colors, target_names)
    plot_risk_coverage_curve(y_train_boot, y_train_boot_proba, y_train_boot_pred, y_test_boot, y_test_boot_proba, y_test_boot_pred)
    mlflow.log_artifact("./plots/class_report.png", artifact_path = "plots")
    mlflow.log_artifact("./plots/calibration_graph.png", artifact_path = "plots")
    mlflow.log_artifact("./plots/risk_coverage.png", artifact_path = "plots")

🏃 View run logistic_regression_model at: http://127.0.0.1:5000/#/experiments/2/runs/3b65a28ccd8049508c820d1ce4679963
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2


In [24]:
normal_dict = pd.DataFrame(all_metrics)
normal_dict["test_accuracy"].quantile([0.025, 0.975])

0.025    0.590535
0.975    0.666667
Name: test_accuracy, dtype: float64

# XGBoost one model

In [26]:
test_size = 0.25
target_names = ["America", "Euro_Africa", "Central_Asia", "East_Asia_Pac"]
colors = ['red', 'green', 'blue', 'black']
num = 1019

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, stratify = y, random_state = num)
X_train = X_train[col_names]
X_test = X_test[col_names]
sample_weights = compute_sample_weight(class_weight='balanced', y=y_train)


with mlflow.start_run(run_name = "xgboost_model"):
    mlflow.set_tag("model_type", "xgboost one model")
    mlflow.set_tag("data type", "24 hour transaction distribution")

    pipeline = xgb.XGBClassifier(
        n_estimators = 926,
        eta = 0.1574838288189856,
        gamma = 7.38464285512075,
        reg_lambda = 7.376833714714335,
        max_depth = 5,
        min_child_weight = 1,
        subsample = 0.6683180715334606,
        colsample_bytree = 0.6711261832095844
    )
    le = LabelEncoder()
    y_train = le.fit_transform(y_train)
    y_test = le.transform(y_test)
    labels = le.classes_
    sample_weights = compute_sample_weight(class_weight='balanced', y=y_train)
    pipeline.fit(X_train, y_train, sample_weight = sample_weights)

    y_train = le.inverse_transform(y_train)
    y_train_pred = np.array(le.inverse_transform(pipeline.predict(X_train)))
    y_train_proba = np.array(pipeline.predict_proba(X_train))

    y_test = le.inverse_transform(y_test)
    y_test_pred = np.array(le.inverse_transform(pipeline.predict(X_test)))
    y_test_proba = np.array(pipeline.predict_proba(X_test))

    metrics = get_mlflow_confidence_metrics(
        y_test, y_test_pred, y_test_proba, 
        y_train, y_train_pred, y_train_proba, 
        pipeline.classes_
    )
    mlflow.log_metrics(metrics)
    class_report(y_train, y_train_pred, y_test, y_test_pred, target_names)
    calibration_curve_classes(y_train, y_train_proba, y_train_pred, y_test, y_test_proba, y_test_pred, colors, target_names)
    plot_risk_coverage_curve(y_train, y_train_proba, y_train_pred, y_test, y_test_proba, y_test_pred)
    mlflow.log_artifact("./plots/class_report.png", artifact_path = "plots")
    mlflow.log_artifact("./plots/calibration_graph.png", artifact_path = "plots")
    mlflow.log_artifact("./plots/risk_coverage.png", artifact_path = "plots")

🏃 View run xgboost_model at: http://127.0.0.1:5000/#/experiments/2/runs/005ebd34fe804c5ca8a0d810c34b381b
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2


# XGBoost bootstraped

In [13]:
n_iterations = 1000
test_size = 0.25
target_names = ["America", "Euro_Africa", "Central_Asia", "East_Asia_Pac"]
colors = ['red', 'green', 'blue', 'black']



with mlflow.start_run(run_name = "xgboost_model"):
    mlflow.set_tag("model_type", "xgboost bootstrapped")
    mlflow.set_tag("data type", "24 hour transaction distribution")
    pipeline = xgb.XGBClassifier(
        n_estimators = 926,
        eta = 0.1574838288189856,
        gamma = 7.38464285512075,
        reg_lambda = 7.376833714714335,
        max_depth = 5,
        min_child_weight = 1,
        subsample = 0.6683180715334606,
        colsample_bytree = 0.6711261832095844
    )
    

    y_test_boot = []
    y_test_boot_pred = []
    y_test_boot_proba = []

    y_train_boot = []
    y_train_boot_pred = []
    y_train_boot_proba = []

    all_metrics = []
    all_class_reports = []

    for i in range(n_iterations):
        print(f"Iteration {i+1}/{n_iterations}")
        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, stratify = y)
        X_train = X_train[col_names]
        X_test = X_test[col_names]
        le = LabelEncoder()
        y_train = le.fit_transform(y_train)
        y_test = le.transform(y_test)
        labels = le.classes_
        sample_weights = compute_sample_weight(class_weight='balanced', y=y_train)
        pipeline.fit(X_train, y_train, sample_weight = sample_weights)

        y_train = le.inverse_transform(y_train)
        y_train_pred = np.array(le.inverse_transform(pipeline.predict(X_train)))
        y_train_proba = np.array(pipeline.predict_proba(X_train))

        y_test = le.inverse_transform(y_test)
        y_test_pred = np.array(le.inverse_transform(pipeline.predict(X_test)))  
        y_test_proba = np.array(pipeline.predict_proba(X_test))

        y_train_boot.extend(y_train)
        y_train_boot_pred.extend(y_train_pred)
        y_train_boot_proba.extend(y_train_proba)

        y_test_boot.extend(y_test)
        y_test_boot_pred.extend(y_test_pred)
        y_test_boot_proba.extend(y_test_proba)

        metrics = get_mlflow_confidence_metrics(
            y_test, y_test_pred, y_test_proba, 
            y_train, y_train_pred, y_train_proba, 
            pipeline.classes_
        )
        all_metrics.append(metrics)

        report_dict = classification_report(y_test, y_test_pred, target_names=target_names, output_dict=True, zero_division=0)
        
        # Flatten the dictionary so it can be easily converted to a DataFrame later
        flat_report = {}
        for label, class_metrics in report_dict.items():
            if isinstance(class_metrics, dict):
                for metric_name, val in class_metrics.items():
                    flat_report[f"{label}_{metric_name}"] = val
            else:
                flat_report[f"{label}"] = class_metrics
                
        all_class_reports.append(flat_report)
    
    y_train_boot = np.array(y_train_boot)
    y_train_boot_pred = np.array(y_train_boot_pred)
    y_train_boot_proba = np.array(y_train_boot_proba)   
    y_test_boot = np.array(y_test_boot) 
    y_test_boot_pred = np.array(y_test_boot_pred)
    y_test_boot_proba = np.array(y_test_boot_proba)

    # Convert the list of dictionaries into a Pandas DataFrame
    df_reports = pd.DataFrame(all_class_reports)
    
    # Calculate Mean, 2.5%, and 97.5% quantiles across the 1000 runs
    report_summary = pd.DataFrame({
        'mean': df_reports.mean(),
        'quantile_2.5%': df_reports.quantile(0.025),
        'quantile_97.5%': df_reports.quantile(0.975)
    })
    
    # Save the quantiles to a CSV and log to MLflow
    csv_path = "./plots/classification_report_quantiles.csv"
    report_summary.to_csv(csv_path)
    mlflow.log_artifact(csv_path, artifact_path="metrics")


    metrics = get_mlflow_confidence_metrics(
        y_test_boot, y_test_boot_pred, y_test_boot_proba, 
        y_train_boot, y_train_boot_pred, y_train_boot_proba, 
        pipeline.classes_
    )
    mlflow.log_metrics(metrics)
    class_report(y_train_boot, y_train_boot_pred, y_test_boot, y_test_boot_pred, target_names)
    calibration_curve_classes(y_train_boot, y_train_boot_proba, y_train_boot_pred, y_test_boot, y_test_boot_proba, y_test_boot_pred, colors, target_names)
    plot_risk_coverage_curve(y_train_boot, y_train_boot_proba, y_train_boot_pred, y_test_boot, y_test_boot_proba, y_test_boot_pred)
    mlflow.log_artifact("./plots/class_report.png", artifact_path = "plots")
    mlflow.log_artifact("./plots/calibration_graph.png", artifact_path = "plots")
    mlflow.log_artifact("./plots/risk_coverage.png", artifact_path = "plots")

Iteration 1/1000
Iteration 2/1000
Iteration 3/1000
Iteration 4/1000
Iteration 5/1000
Iteration 6/1000
Iteration 7/1000
Iteration 8/1000
Iteration 9/1000
Iteration 10/1000
Iteration 11/1000
Iteration 12/1000
Iteration 13/1000
Iteration 14/1000
Iteration 15/1000
Iteration 16/1000
Iteration 17/1000
Iteration 18/1000
Iteration 19/1000
Iteration 20/1000
Iteration 21/1000
Iteration 22/1000
Iteration 23/1000
Iteration 24/1000
Iteration 25/1000
Iteration 26/1000
Iteration 27/1000
Iteration 28/1000
Iteration 29/1000
Iteration 30/1000
Iteration 31/1000
Iteration 32/1000
Iteration 33/1000
Iteration 34/1000
Iteration 35/1000
Iteration 36/1000
Iteration 37/1000
Iteration 38/1000
Iteration 39/1000
Iteration 40/1000
Iteration 41/1000
Iteration 42/1000
Iteration 43/1000
Iteration 44/1000
Iteration 45/1000
Iteration 46/1000
Iteration 47/1000
Iteration 48/1000
Iteration 49/1000
Iteration 50/1000
Iteration 51/1000
Iteration 52/1000
Iteration 53/1000
Iteration 54/1000
Iteration 55/1000
Iteration 56/1000
I